# A05 · 多广告位GFP、GSP与VCG

先修：公用31–34；A04。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

采用可分离点击模型P(click|i,k)=α_k，所有广告质量相同，估值/报价单位为每点击货币；广告位α=(1,.4)，第三位不展示但提供竞争。GFP每点击付自己报价；GSP付下一位报价；VCG按移除赢家后他人的最优福利减去原先他人福利收费。输出payment是每请求期望支付，CPC=payment/α，不可混用。效率只在此可分离模型及真实估值排序下成立。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
for mechanism in ('gfp','gsp','vcg'):
 print(mechanism);display(pd.DataFrame(ac.slots(mechanism=mechanism)))
vcg=ac.slots();assert np.allclose([r['payment'] for r in vcg],[4.,.4])
from itertools import permutations
assert max(1*[7,6,1][p[0]]+.4*[7,6,1][p[1]] for p in permutations(range(3),2))==9.4
def utility(b,mechanism):
 row=next((r for r in ac.slots((b,6,1),mechanism=mechanism) if r['bidder']==0),None)
 return 7*row['click_probability']-row['payment'] if row else 0.
grid=np.arange(0,10,.25)
assert max(utility(b,'vcg') for b in grid)<=utility(7,'vcg')+1e-10
print('GSP truthful/shaded utility',utility(7,'gsp'),utility(5,'gsp'))
assert utility(5,'gsp')>utility(7,'gsp')

gfp


,bidder,slot,click_probability,payment,cpc
0,0,0,1.0,7.0,7.0
1,1,1,0.4,2.4,6.0


gsp


,bidder,slot,click_probability,payment,cpc
0,0,0,1.0,6.0,6.0
1,1,1,0.4,0.4,1.0


vcg


,bidder,slot,click_probability,payment,cpc
0,0,0,1.0,4.0,4.0
1,1,1,0.4,0.4,1.0


GSP truthful/shaded utility 1.0 2.4000000000000004


## 解释与局限

VCG按外部性计费，顶部期望支付4而非下一位CPC 6。GSP在此例降低报价可能增加自身效用，所以单物品二价的诚实结论不能直接移植。网格断言只验证此例；一般VCG激励结论还依赖准线性效用、独立私人价值、无预算等假设。

## 查看源码，再做练习

把第二位置α从.4改为.8并重算支付。添加质量q_i后推导质量调整GSP阈值，不要继续直接取下一报价。

In [3]:
import inspect
print(inspect.getsource(ac.slots))

def slots(bids=(7.,6.,1.), alpha=(1.,.4), mechanism='vcg'):
    """Homogeneous quality, separable position CTR alpha, bids in currency/click.
    payment is expected currency per request, not CPC. Zero reserve; stable ties.
    """
    bids=np.asarray(bids,float); alpha=np.asarray(alpha,float)
    if mechanism not in ('gfp','gsp','vcg') or len(bids)<len(alpha) or np.any(bids<0) or np.any(np.diff(alpha)>0) or np.any(alpha<=0):
        raise ValueError('Invalid auction')
    order=sorted(range(len(bids)),key=lambda i:(-bids[i],i)); winners=order[:len(alpha)]
    welfare=sum(alpha[k]*bids[i] for k,i in enumerate(winners)); rows=[]
    for k,i in enumerate(winners):
        if mechanism=='gfp': pay=alpha[k]*bids[i]
        elif mechanism=='gsp': pay=alpha[k]*(bids[order[k+1]] if k+1<len(order) else 0.)
        else:
            other=[j for j in order if j!=i][:len(alpha)]
            without=sum(alpha[t]*bids[j] for t,j in enumerate(other))
            pay=without-(welfare-alpha[k]*bids[i

## 来源与下一步

A-CCR 73–94页；A-RTB 21–36页；不沿用课件历史平台现状描述。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。